In [1]:
import pandas as pd

meta = pd.read_csv('/scratch/leuven/387/vsc38795/postcard_color_project/20230301-Postcards-csv.csv')
print(meta['Resolver URL 856$u'].head(5).tolist())

['http://resolver.libis.be/IE2777387/representation', 'http://resolver.libis.be/IE2783050/representation', 'http://resolver.libis.be/IE2785913/representation', 'http://resolver.libis.be/IE2786767/representation', 'http://resolver.libis.be/IE2788279/representation']


In [2]:
import pandas as pd

labels = pd.read_csv('/scratch/leuven/387/vsc38795/postcard_color_project/output/postcard_fine_labels.csv')
meta = pd.read_csv('/scratch/leuven/387/vsc38795/postcard_color_project/20230301-Postcards-csv.csv')

meta = meta.rename(columns={
    'Resolver URL 856$u':                   'image_url',
    'Date 264$c_estimateDecade':            'decade',
    'Uniform title 130$a_placeNameCurrent': 'city',
    'Main title 245$a':                     'title',
    'Publisher 264$b_clean':                'publisher'
})
meta['IE_id'] = meta['image_url'].str.extract(r'/(IE\d+)/')
labels['IE_id'] = labels['image_path'].str.extract(r'/(IE\d+)/')

df = labels.merge(
    meta[['IE_id', 'image_url', 'decade', 'city', 'title', 'publisher']],
    on='IE_id', how='left'
)

print(f"Total: {len(df)}")
print(df[['fine_label','decade','city','image_url']].head(3))

df.to_csv(
    '/scratch/leuven/387/vsc38795/postcard_color_project/output/postcard_recommendation_db.csv',
    index=False
)
print("Saved.")

Total: 35930
  fine_label  decade     city  \
0         bw  1900's  Lokeren   
1         bw  1920's  Lokeren   
2         bw  1900's  Lokeren   

                                           image_url  
0  http://resolver.libis.be/IE10071724/representa...  
1  http://resolver.libis.be/IE10071745/representa...  
2  http://resolver.libis.be/IE10071750/representa...  
Saved.


In [3]:
import pandas as pd
import json

df = pd.read_csv('/scratch/leuven/387/vsc38795/postcard_color_project/output/postcard_recommendation_db.csv')

# 只保留網頁需要的欄位
df_web = df[[
    'IE_id', 'fine_label', 'decade', 'city', 
    'title', 'publisher', 'image_url'
]].copy()

# 把 thumbnail URL 加進去
df_web['thumbnail_url'] = df_web['image_url'].str.replace(
    'representation', 'thumbnail', regex=False
)

# 移除缺值
df_web = df_web.dropna(subset=['image_url'])

print(f"Total records: {len(df_web)}")
print(df_web.head(2).to_string())

# 存成 JSON
df_web.to_json(
    '/scratch/leuven/387/vsc38795/postcard_color_project/output/postcards.json',
    orient='records',
    force_ascii=False,
    indent=2
)
print("Saved postcards.json")

Total records: 35648
        IE_id fine_label  decade     city                                                                 title                   publisher                                           image_url                                  thumbnail_url
0  IE10071724         bw  1900's  Lokeren  Lokeren. Souvenir du cortège fleuri (Elèves de la présentation N.D.)                     Hendrix  http://resolver.libis.be/IE10071724/representation  http://resolver.libis.be/IE10071724/thumbnail
1  IE10071745         bw  1920's  Lokeren                                        Lokeren. Vue sur la Durme [01]  [publisher not identified]  http://resolver.libis.be/IE10071745/representation  http://resolver.libis.be/IE10071745/thumbnail
Saved postcards.json


In [4]:
import pandas as pd

df = pd.read_csv('/scratch/leuven/387/vsc38795/postcard_color_project/output/postcard_recommendation_db.csv')

df_web = df[['IE_id', 'fine_label', 'decade', 'city', 'title', 'publisher', 'image_url']].copy()
df_web['thumbnail_url'] = df_web['image_url'].str.replace('representation', 'thumbnail', regex=False)
df_web = df_web.dropna(subset=['image_url'])

print(f"Total records: {len(df_web)}")
print("\nUnique decades:", sorted(df_web['decade'].dropna().unique().tolist()))
print("\nUnique fine_labels:", sorted(df_web['fine_label'].dropna().unique().tolist()))
print("\nTop 10 cities:", df_web['city'].value_counts().head(10).index.tolist())

Total records: 35648

Unique decades: ["1890's", "1900's", "1910's", "1920's", "1930's", "1940's", "1950's", "1960's", "1970's", "1980's", "1990's"]

Unique fine_labels: ['bw', 'color_handcolored', 'color_photo', 'monotone_blue', 'monotone_green', 'monotone_purple', 'monotone_red', 'sepia']

Top 10 cities: ['Antwerpen', 'Brussel-Bruxelles', 'Brugge', 'Gent', 'Liège', 'Mechelen', 'Namur', 'Oostende', 'Dinant', 'Knokke-Heist']


In [5]:
df_web.to_json(
    '/scratch/leuven/387/vsc38795/postcard_color_project/output/postcards.json',
    orient='records',
    force_ascii=False,
    indent=2
)
print("Saved.")

Saved.
